# Equity Duration × ECB Monetary Policy Shocks

This notebook studies how **firm-level equity duration** shapes stock price reactions to **ECB monetary policy announcements**, using a high-frequency event-study design.

The central hypothesis is that **equities with longer duration—i.e. firms whose value is concentrated in distant future cash flows—are more sensitive to unexpected changes in discount rates**. Monetary policy surprises provide a natural laboratory to test this prediction, as they induce abrupt shifts in interest rates and risk premia that are plausibly exogenous to individual firms.

A key contribution of this notebook is the introduction and empirical implementation of a **novel, net-payout-based equity duration measure**, which serves as the **primary duration concept** throughout the analysis.

---

## Research Question

Do **high-duration equities react more strongly** to ECB monetary policy surprises than **low-duration equities**, and does this sensitivity differ across **policy shocks** and **information shocks**?

More specifically, this notebook addresses three questions:

1. Does equity duration amplify firm-level stock price reactions to unexpected ECB policy shocks?
2. Is duration sensitivity driven primarily by **discount-rate shocks** rather than **information shocks**?
3. How does the explanatory power of a **net-payout-based equity duration** compare to standard DCF-based duration measures?

---

## Measuring Equity Duration

### Primary Duration Measure: Net-Payout-Based Equity Duration

The primary duration measure in this notebook is a **net-payout-based equity duration**, constructed from a forward-looking valuation identity that links **equity prices, expected payouts, and discount rates**.

Rather than discounting exogenously specified cash flows at an assumed cost of capital, this approach infers the **timing of equity cash flows directly from market valuations**. Expected net payouts—defined as dividends plus share repurchases minus equity issuances—are forecast using firm-level state variables, and the firm-specific discount rate is recovered endogenously such that the present value of expected payouts matches the observed equity valuation.

Formally, equity duration is defined as the **payout-weighted average maturity of expected net payouts**, using the same discount rate that rationalizes the firm’s market-to-book ratio. This ensures internal consistency between prices, expected cash flows, and discounting.

This measure has several advantages:

- It is **forward-looking** and disciplined by observed equity prices.
- It captures both **cash-flow timing** and **discount-rate exposure** in a model-consistent way.
- It is directly tied to **shareholder-relevant cash flows**, rather than accounting-based proxies.
- It avoids imposing auxiliary asset-pricing assumptions such as a fixed discount rate or a CAPM structure.

Because this duration measure is constructed **ex ante** and remains fixed within each event window, it is well suited for causal analysis of stock price reactions to monetary policy shocks.

---

### Benchmark Duration Measures

To assess robustness and disentangle timing versus discounting effects, I compare the net-payout-based duration to two commonly used benchmark measures:

- **`Duration_CAPM`**  
  A discounted-cash-flow (DCF) equity duration that applies a **firm-specific CAPM-implied cost of equity**.  
  This measure captures both cash-flow timing and heterogeneity in required returns driven by systematic risk.

- **`Duration_r125`**  
  A DCF-based equity duration computed using a **constant discount rate of 12.5% for all firms**.  
  By fixing the discount rate, this measure isolates **pure cash-flow timing**, abstracting from cross-sectional variation in discount rates.

All duration measures are standardized prior to estimation. Comparing their performance allows me to assess whether monetary policy transmission operates primarily through **cash-flow horizons**, **discount-rate heterogeneity**, or a combination of both.

---

## ECB Monetary Policy Shocks

The analysis focuses on ECB announcement days and exploits a high-frequency identification strategy that decomposes monetary policy surprises into two orthogonal components:

- **`ShockMP`**: a *pure monetary policy shock*, interpreted as an unexpected tightening or easing that primarily affects discount rates.
- **`ShockInfo`**: an *information shock*, capturing news about the economic outlook conveyed by the ECB that moves yields and equities in the same direction.

This decomposition is crucial, as **equity prices may respond differently to discount-rate news than to information about future cash flows**. Duration-based predictions apply most cleanly to the former.

---

## Empirical Design

For each ECB announcement date, I compute **firm-level abnormal returns (AR)** and estimate panel regressions of the form:

$$
AR_{i,t} = \beta_0 + \beta_1 Shock_t + \beta_2 D_i + \beta_3 (Shock_t \times D_i) + \Gamma'X_i + \varepsilon_{i,t},
$$

where $D_i$ denotes a standardized equity duration measure and $X_i$ includes firm characteristics such as size and market beta.

The coefficient of interest is the interaction term $ \beta_3 $, which captures whether **equity duration amplifies firms’ sensitivity to monetary policy shocks**. Standard errors are **clustered by event date**, reflecting the fact that shocks are common across firms within each ECB meeting.

In extended specifications, I replace $Shock_t$ with `ShockMP` and `ShockInfo` to separately identify **discount-rate** and **information** channels of monetary transmission.

---

## Interpretation

Under the duration hypothesis, **unexpected policy tightenings should depress the prices of high-duration equities more strongly**, as their value depends disproportionately on distant cash flows that are more sensitive to increases in discount rates. By contrast, information shocks may have weaker or ambiguous duration effects, depending on how news about future economic conditions affects long-run cash-flow expectations.

By combining a novel net-payout-based duration measure with high-frequency ECB shock identification, this notebook provides a clean empirical test of the role of equity duration in monetary policy transmission.

In [28]:
# Core
import numpy as np
import pandas as pd
from pathlib import Path
# Stats / regressions
import statsmodels.formula.api as smf
# Plots
import matplotlib.pyplot as plt

#Speicherstruktur für Intermediate und Final Output
BASE_DIR = Path(
    "/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data"
)
BASE_DIR.mkdir(parents=True, exist_ok=True)

(TABLE_DIR := BASE_DIR / "tables").mkdir(exist_ok=True)
(DATA_DIR  := BASE_DIR / "intermediate").mkdir(exist_ok=True)

DUR_FILE = TABLE_DIR / "final_results_duration.xlsx"
SHOCK_FILE = TABLE_DIR / "shocks_ecb_mpd_me_d.csv"

def save_parquet(df: pd.DataFrame, name: str):
    path = DATA_DIR / f"{name}.parquet"
    df.to_parquet(path, index=False)
    print(f"Saved: {path}")

def load_parquet(name: str) -> pd.DataFrame:
    path = DATA_DIR / f"{name}.parquet"
    return pd.read_parquet(path)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

## 2. Load firm-level equity duration and characteristics

I load the firm-level duration measures and keep **Duration (CAPM r)** as the baseline equity-duration proxy.


In [29]:
# Load durations
df_dur_panel = load_parquet("equity_duration_panel")  # columns: RIC | CompanyName | YEAR | Duration_*

# Harmonize column names
if "YEAR" not in df_dur_panel.columns and "year" in df_dur_panel.columns:
    df_dur_panel = df_dur_panel.rename(columns={"year": "YEAR"})

# Basic cleaning
df_dur_panel["RIC"] = df_dur_panel["RIC"].astype(str).str.strip()
df_dur_panel["YEAR"] = pd.to_numeric(df_dur_panel["YEAR"], errors="coerce").astype("Int64")

# Choose baseline duration measure for the main analysis
DUR_COL = "Duration_NP"     
df_dur_panel[DUR_COL] = pd.to_numeric(df_dur_panel[DUR_COL], errors="coerce")

display(df_dur_panel[["RIC", "CompanyName", "YEAR", DUR_COL]].head())

,RIC,CompanyName,YEAR,Duration_NP
0,1910.HK,Samsonite Group SA,2012,92.581988
1,1910.HK,Samsonite Group SA,2013,100.119876
2,1910.HK,Samsonite Group SA,2014,109.052811
3,1910.HK,Samsonite Group SA,2015,102.856076
4,1910.HK,Samsonite Group SA,2016,99.026306


## 3. Load ECB shock series


### Economic Meaning and Sign Conventions

I use the Jarociński–Karadi (2020) decomposition of high-frequency ECB surprises into two orthogonal components:

- **`ShockMP` (Monetary Policy Shock):** the “pure” policy component.  
  A **positive** value corresponds to an unexpected **tightening** (higher-than-expected short rates).  
  **Prediction:** on average, equity prices fall → abnormal returns should be negative.

- **`ShockInfo` (Information Shock):** captures the information revealed by the central bank about the outlook.  
  A **positive** value is interpreted as **good news** about fundamentals (often associated with rising yields and rising equities).  
  **Prediction:** equity prices may rise, depending on the balance of improved cash-flow expectations vs discount-rate effects.

I run both a one-shock (MP only) specification and a two-shock specification to separately identify discount-rate vs information channels.

In [30]:
df_shk = pd.read_csv(SHOCK_FILE)

# ------------------------------------------------------------
# 1. Parse dates
# ------------------------------------------------------------
df_shk["date"] = pd.to_datetime(df_shk["date"], errors="coerce")
assert df_shk["date"].notna().all(), "Some ECB shock dates could not be parsed."

# ------------------------------------------------------------
# 2. Select shock decomposition
# ------------------------------------------------------------
# Available decompositions:
#
#   - "median" : General decomposition (baseline)
#                -> both monetary policy and information shocks
#                   can occur jointly within an announcement
#
#   - "pm"     : Restrictive decomposition (robustness)
#                -> mutually exclusive shocks ("poor man's" sign restrictions)
#
# Baseline choice for the paper: "median"

SHOCK_VERSION = "median"   # <-- BASELINE (recommended)

if SHOCK_VERSION == "median":
    shock_map = {
        "MP_median": "ShockMP",
        "CBI_median": "ShockInfo"
    }
elif SHOCK_VERSION == "pm":
    shock_map = {
        "MP_pm": "ShockMP",
        "CBI_pm": "ShockInfo"
    }
else:
    raise ValueError("SHOCK_VERSION must be 'median' or 'pm'.")

missing = [c for c in shock_map if c not in df_shk.columns]
if missing:
    raise ValueError(f"Missing required shock columns: {missing}")

df_shk = df_shk.rename(columns=shock_map)

# ------------------------------------------------------------
# 3. Optional: keep policy factor (pc1) for diagnostics only
# ------------------------------------------------------------
if "pc1" in df_shk.columns:
    df_shk["PolicyFactor"] = pd.to_numeric(df_shk["pc1"], errors="coerce")

# ------------------------------------------------------------
# 4. Keep one observation per ECB announcement date
# ------------------------------------------------------------
df_shk = (
    df_shk
    .drop_duplicates(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 5. Numeric coercion
# ------------------------------------------------------------
for c in ["ShockMP", "ShockInfo"]:
    df_shk[c] = pd.to_numeric(df_shk[c], errors="coerce")

# ------------------------------------------------------------
# 6. Consistency check
# ------------------------------------------------------------
# NOTE:
# - For the restrictive decomposition ("pm"), MP + Info == pc1 holds by construction
# - For the general decomposition ("median"), this equality does NOT need to hold
#   and deviations are expected and economically meaningful

if "PolicyFactor" in df_shk.columns and SHOCK_VERSION == "pm":
    check = df_shk["ShockMP"] + df_shk["ShockInfo"] - df_shk["PolicyFactor"]
    print("Max |MP + Info − pc1| (pm):", check.abs().max())

# ------------------------------------------------------------
# 7. Final sanity output
# ------------------------------------------------------------
print("Shock decomposition used:", SHOCK_VERSION)
print("Number of ECB events:", df_shk.shape[0])

display(df_shk[["date", "ShockMP", "ShockInfo"]].head())

Shock decomposition used: median
Number of ECB events: 312


,date,ShockMP,ShockInfo
0,1999-01-07,0.020578,-0.058123
1,1999-01-21,0.008569,-0.004988
2,1999-02-18,-0.005565,0.005565
3,1999-03-04,-0.003596,0.001670
4,1999-03-18,-0.002326,0.001568


## 4. Event Panel Shocks x Duratons

### Load daily returns and construct abnormal returns (AR)

My baseline dependent variable is the **abnormal return** on each firm-event date.  
(Depending on the earlier construction in the notebook, this is either a market-adjusted return or a model-based abnormal return.)

To avoid relying on external index data (e.g., STOXX), I compute the daily market return directly from my sample:

$$
mkt\_ret_t = \frac{1}{N_t}\sum_i r_{i,t}
$$

and define abnormal returns as:

$$
AR_{i,t} = r_{i,t} - mkt\_ret_t
$$

This is a standard equal-weighted benchmark in event-study applications.

### Build the event-day panel (firm × ECB event date)

I keep only returns observed on ECB event dates and merge:
- shocks by `date`
- duration and firm controls by `RIC`

This creates the core event-study panel used in the regressions.

### Event Windows
- **Day 0:** the ECB announcement day (baseline window).
- **[0,+1] window:** a robustness check that sums Day 0 and Day +1 abnormal returns to capture delayed price adjustment.

Interpretation: the shock occurs on Day 0, but markets may incorporate information with a short delay, so [0,+1] provides a more conservative reaction measure.

### Standardize equity duration

I standardize duration to mean 0 and standard deviation 1:

$$
D^{std}_i = \frac{D_i - \bar{D}}{sd(D)}
$$

This makes interaction coefficients easy to interpret as the effect per **one standard deviation** increase in duration.



In [31]:
# ============================================================
# Load daily returns (firm × day) and ensure AR exists
# ============================================================

df_ret = load_parquet("returns_daily")

# --- basic checks ---
assert "RIC" in df_ret.columns, "RIC missing in returns_daily"
assert "date" in df_ret.columns, "date missing in returns_daily"

df_ret["RIC"] = df_ret["RIC"].astype(str).str.strip()
df_ret["date"] = pd.to_datetime(df_ret["date"], errors="coerce")
assert df_ret["date"].notna().any(), "All dates are NaT after parsing."

print("returns_daily shape:", df_ret.shape)
print("Columns:", df_ret.columns.tolist()[:50])

# --- 1) Identify a daily return column if AR is missing ---
if "AR" not in df_ret.columns:
    # candidate names for raw daily returns
    candidates = [c for c in df_ret.columns if c.lower() in [
        "ret", "return", "returns", "daily_return", "tr.totalreturn", "tr.totalreturn1d",
        "tr.totalreturngross", "tr.pricepctchg", "pct_change", "r"
    ]]

    if len(candidates) == 0:
        # fallback heuristic: any numeric column with "ret" in name
        candidates = [c for c in df_ret.columns if ("ret" in c.lower() or "return" in c.lower())]

    if len(candidates) == 0:
        raise ValueError(
            "AR is missing and I could not infer a daily return column.\n"
            "Please tell me which column in returns_daily is the daily return."
        )

    RET_COL = candidates[0]
    print(f"[Info] AR not found. Using '{RET_COL}' as raw daily return column.")

    df_ret[RET_COL] = pd.to_numeric(df_ret[RET_COL], errors="coerce")

    # --- 2) Build a simple market-adjusted abnormal return: AR = ret - cross-sectional mean ret on that day ---
    # (This matches your earlier approach and is fine for cross-sectional heterogeneity.)
    mkt_ret = (
        df_ret.groupby("date")[RET_COL]
        .mean()
        .rename("mkt_ret")
        .reset_index()
    )

    df_ret = df_ret.merge(mkt_ret, on="date", how="left", validate="m:1")
    df_ret["AR"] = df_ret[RET_COL] - df_ret["mkt_ret"]

    print("[Info] Constructed AR = return - daily cross-sectional mean return.")

# --- final sanity checks ---
assert "AR" in df_ret.columns, "AR still missing after construction."
print("AR summary:")
display(df_ret["AR"].describe())
display(df_ret[["RIC", "date", "AR"]].head())


returns_daily shape: (5369859, 3)
Columns: ['date', 'RIC', 'ret']
[Info] AR not found. Using 'ret' as raw daily return column.
[Info] Constructed AR = return - daily cross-sectional mean return.
AR summary:


count      5369859.0
mean            -0.0
std        30.445973
min      -170.560539
25%        -1.041264
50%        -0.090752
75%         0.902584
max      41899.11531
Name: AR, dtype: Float64

,RIC,date,AR
0,1910.HK,2011-06-17,0.354292
1,1910.HK,2011-06-20,0.983604
2,1910.HK,2011-06-21,2.115562
3,1910.HK,2011-06-22,0.414274
4,1910.HK,2011-06-23,4.906022


In [32]:
# ============================================================
# Build firm × event-date panel and merge ECB shocks + durations
# + standardize each duration (new *_std columns)
# ============================================================

# --- 1) Event dates from shocks ---
df_shk["date"] = pd.to_datetime(df_shk["date"], errors="coerce")
assert df_shk["date"].notna().all(), "Some shock dates could not be parsed."

event_dates = df_shk["date"].dropna().sort_values().unique()
print("Number of event dates:", len(event_dates))

# --- 2) Detect date column in df_ret ---
if "date" in df_ret.columns:
    DATE_COL = "date"
else:
    candidates = [c for c in df_ret.columns if c.lower() in ["day", "tradedate", "event_date", "datadate", "datetime"]]
    if len(candidates) == 0:
        raise ValueError(
            "Could not find a date column in df_ret. "
            "Rename your date column to 'date' or set DATE_COL manually."
        )
    DATE_COL = candidates[0]

print("Using df_ret date column:", DATE_COL)

# Parse dates in df_ret
df_ret[DATE_COL] = pd.to_datetime(df_ret[DATE_COL], errors="coerce")
assert df_ret[DATE_COL].notna().any(), "All df_ret dates are NaT after parsing."

# --- 3) Build event panel ---
df_evt = df_ret[df_ret[DATE_COL].isin(event_dates)].copy()
df_evt = df_evt.rename(columns={DATE_COL: "date"})
df_evt["RIC"] = df_evt["RIC"].astype(str).str.strip()

print("df_evt shape (firm × event dates):", df_evt.shape)
display(df_evt.head())

# --- 4) Merge shocks ---
df_evt = df_evt.merge(
    df_shk[["date", "ShockMP", "ShockInfo"]],
    on="date",
    how="left",
    validate="m:1"
)

miss_shk = df_evt[["ShockMP", "ShockInfo"]].isna().any(axis=1).mean()
print("Share of rows with missing shocks:", miss_shk)
assert miss_shk == 0, "Some event rows could not be matched to shocks."

# --- 5) Construct predetermined YEAR for duration merge (use year-end t-1) ---
df_evt["YEAR"] = (df_evt["date"].dt.year - 1).astype("Int64")

# --- 6) Merge firm-year numeric vars by RIC x YEAR ---
df_dur_panel["RIC"] = df_dur_panel["RIC"].astype(str).str.strip()

# harmonize year column name in df_dur_panel
if "YEAR" not in df_dur_panel.columns and "year" in df_dur_panel.columns:
    df_dur_panel = df_dur_panel.rename(columns={"year": "YEAR"})

assert "YEAR" in df_dur_panel.columns, "df_dur_panel has no YEAR/year column."

df_dur_panel["YEAR"] = pd.to_numeric(df_dur_panel["YEAR"], errors="coerce").astype("Int64")

duration_cols = [c for c in df_dur_panel.columns if c.startswith("Duration_")]
num_cols = [c for c in (duration_cols + ["BETA_5Y", "ME"]) if c in df_dur_panel.columns]

for c in num_cols:
    df_dur_panel[c] = pd.to_numeric(df_dur_panel[c], errors="coerce")

df_evt = df_evt.merge(
    df_dur_panel[["RIC", "YEAR"] + num_cols].drop_duplicates(["RIC", "YEAR"]),
    on=["RIC", "YEAR"],
    how="left",
    validate="m:1"
)

# --- 6b) Merge sectors by RIC only (time-invariant) ---
sector_cols = [c for c in ["EconomicSector", "BusinessSector"] if c in df_dur_panel.columns]
sector_map = df_dur_panel[["RIC"] + sector_cols].drop_duplicates("RIC")

df_evt = df_evt.merge(sector_map, on="RIC", how="left", validate="m:1")

# --- 6b) Standardize each duration into a separate *_std column (z-score) ---
# Standardization is done on the merged sample (df_evt), ignoring NaNs.

std_cols = []
for c in duration_cols:
    mu = df_evt[c].mean(skipna=True)
    sd = df_evt[c].std(skipna=True, ddof=0)  # ddof=0 -> population std; ddof=1 if sample std
    new_c = f"{c}_std"
    std_cols.append(new_c)

    if pd.isna(sd) or sd == 0:
        df_evt[new_c] = pd.NA
    else:
        df_evt[new_c] = (df_evt[c] - mu) / sd

print("Standardized duration columns added:", std_cols)

# --- 7) Diagnostics ---
print("\nMissing share by duration column:")
display(df_evt[duration_cols].isna().mean().sort_values())

print("\nMissing share by standardized duration column:")
display(df_evt[std_cols].isna().mean().sort_values())

print("\nPreview merged columns:")
display(df_evt[["RIC", "date", "YEAR", "ShockMP", "ShockInfo"] + duration_cols + std_cols].head(10))

Number of event dates: 312
Using df_ret date column: date
df_evt shape (firm × event dates): (226233, 5)


,date,RIC,ret,mkt_ret,AR
13,2011-07-07,1910.HK,0.277008,0.685079,-0.408071
33,2011-08-04,1910.HK,-0.374065,-3.637365,3.2633
58,2011-09-08,1910.HK,-1.540832,0.193622,-1.734454
75,2011-10-06,1910.HK,10.410095,2.143145,8.26695
95,2011-11-03,1910.HK,-5.769231,1.670973,-7.440204


Share of rows with missing shocks: 0.0
Standardized duration columns added: ['Duration_NP_std', 'Duration_NP_w_std', 'Duration_undiscounted_std', 'Duration_r125_std', 'Duration_CAPM_std', 'Duration_emp_2yOIS_std']

Missing share by duration column:


Duration_NP              0.255073
Duration_NP_w            0.255073
Duration_emp_2yOIS       0.268356
Duration_r125            0.435250
Duration_undiscounted    0.435414
Duration_CAPM            0.461224
dtype: float64


Missing share by standardized duration column:


Duration_NP_std              0.255073
Duration_NP_w_std            0.255073
Duration_emp_2yOIS_std       0.268356
Duration_r125_std            0.435250
Duration_undiscounted_std    0.435414
Duration_CAPM_std            0.461224
dtype: float64


Preview merged columns:


,RIC,date,YEAR,ShockMP,ShockInfo,Duration_NP,Duration_NP_w,Duration_undiscounted,Duration_r125,Duration_CAPM,Duration_emp_2yOIS,Duration_NP_std,Duration_NP_w_std,Duration_undiscounted_std,Duration_r125_std,Duration_CAPM_std,Duration_emp_2yOIS_std
0,1910.HK,2011-07-07,2010,-0.013549,0.027702,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1910.HK,2011-08-04,2010,-0.043663,-0.120139,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1910.HK,2011-09-08,2010,0.012075,-0.010010,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1910.HK,2011-10-06,2010,0.106199,0.050311,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1910.HK,2011-11-03,2010,-0.071923,-0.061193,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1910.HK,2011-12-08,2010,0.033493,-0.039483,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1910.HK,2012-01-12,2011,0.026299,-0.028402,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,1910.HK,2012-02-09,2011,-0.014338,0.015358,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,1910.HK,2012-03-08,2011,0.006830,-0.008305,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,1910.HK,2012-05-03,2011,0.041357,-0.026215,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [33]:
# -----------------------------
# 1) Load Euro500 + clean
# -----------------------------
euro500 = pd.read_parquet(DATA_DIR / "euro500.parquet").copy()
euro500["date"] = pd.to_datetime(euro500["date"], errors="coerce").dt.normalize()
euro500 = euro500.dropna(subset=["RIC", "date"]).copy()
euro500["RIC"] = euro500["RIC"].astype(str).str.strip()

# membership key
euro500_key = euro500[["RIC", "date"]].drop_duplicates().copy()
euro500_key["in_euro500"] = 1

# -----------------------------
# 2) Clean df_evt dates + RIC
# -----------------------------
df_evt = df_evt.copy()
df_evt["date"] = pd.to_datetime(df_evt["date"], errors="coerce").dt.normalize()
df_evt["RIC"]  = df_evt["RIC"].astype(str).str.strip()

# -----------------------------
# 3) Map event-date -> last quarter end (<= event date)
# -----------------------------
qend = pd.offsets.QuarterEnd()
df_evt["q_end"] = df_evt["date"].apply(lambda d: qend.rollback(d) if pd.notna(d) else pd.NaT)

# -----------------------------
# 4) Merge membership using q_end
# -----------------------------
df_evt = df_evt.merge(
    euro500_key.rename(columns={"date": "q_end"}),
    on=["RIC", "q_end"],
    how="left"
)

df_evt["in_euro500"] = df_evt["in_euro500"].fillna(0).astype(int)

print("Share in Euro500 (rows):", df_evt["in_euro500"].mean())
print("Unique Euro500 RICs in df_evt:", df_evt.loc[df_evt["in_euro500"] == 1, "RIC"].nunique())

Share in Euro500 (rows): 0.5148497345656912
Unique Euro500 RICs in df_evt: 1024


In [34]:
# Alle Duration-Spalten finden (passt zu deinem Naming)
DUR_COLS = [c for c in df_evt.columns if "Duration" in c]

# Duration auf NaN setzen, wenn nicht im Euro500
df_evt.loc[df_evt["in_euro500"] == 0, DUR_COLS] = np.nan

print("Durations set to NaN outside Euro500.")
print("Non-missing duration share:", df_evt[DUR_COLS].notna().any(axis=1).mean())

Durations set to NaN outside Euro500.
Non-missing duration share: 0.39461528601044055


In [35]:
# ------------------------------------------------------------
# CHECK: per event date, max 500 Euro500 members / durations
# ------------------------------------------------------------

# 1) Wie viele Euro500-Members pro Event?
cnt_members = (
    df_evt.loc[df_evt["in_euro500"] == 1]
    .groupby("date")["RIC"]
    .nunique()
)

print("Euro500 members per event (nunique RIC):")
print(cnt_members.describe())

viol_members = cnt_members[cnt_members > 500]
print(f"\nEvents with >500 Euro500 members: {len(viol_members)}")
if len(viol_members) > 0:
    print(viol_members.head(20))

# 2) Wie viele Durations pro Event? (wähle die Duration-Spalte, die du nutzt)
BASE_DUR = "Duration_NP_std"   # <- anpassen falls nötig
assert BASE_DUR in df_evt.columns, f"{BASE_DUR} not in df_evt"

cnt_dur = (
    df_evt.loc[(df_evt["in_euro500"] == 1) & (df_evt[BASE_DUR].notna())]
    .groupby("date")["RIC"]
    .nunique()
)

print("\nNon-missing duration (Euro500 only) per event:")
print(cnt_dur.describe())

viol_dur = cnt_dur[cnt_dur > 500]
print(f"\nEvents with >500 non-missing durations: {len(viol_dur)}")
if len(viol_dur) > 0:
    print(viol_dur.head(20))

# 3) Optional: harte OK/FAIL Meldung
ok_members = (cnt_members.max() <= 500) if len(cnt_members) else True
ok_dur = (cnt_dur.max() <= 500) if len(cnt_dur) else True

print("\nCHECK RESULT:")
print(" - Members <= 500:", "OK" if ok_members else "FAIL")
print(" - Durations <= 500:", "OK" if ok_dur else "FAIL")

Euro500 members per event (nunique RIC):
count    312.000000
mean     373.317308
std       71.804164
min      264.000000
25%      298.000000
50%      385.500000
75%      432.250000
max      500.000000
Name: RIC, dtype: float64

Events with >500 Euro500 members: 0

Non-missing duration (Euro500 only) per event:
count    312.000000
mean     286.134615
std      105.293776
min       81.000000
25%      208.000000
50%      322.500000
75%      366.000000
max      450.000000
Name: RIC, dtype: float64

Events with >500 non-missing durations: 0

CHECK RESULT:
 - Members <= 500: OK
 - Durations <= 500: OK


## Why I Use Predetermined Duration (YEAR − 1)

To avoid **look-ahead bias**, I merge **firm-year duration measures from the previous year** to each ECB event.  
Concretely, for an ECB announcement in calendar year *t*, I use the equity duration measured at **year-end (12/31) of year *t−1***.

This ensures that duration is **known before the event** and cannot be mechanically affected by event-day returns.

**Implementation:**  
`YEAR = year(date) − 1` and merge on (`RIC`, `YEAR`).

## 7. Baseline regression: monetary policy shock × duration

### Core Hypotheses and Empirical Specification

This chapter studies heterogeneous stock price responses to ECB monetary policy announcements using a **two-shock framework** that explicitly separates **monetary policy shocks** from **central bank information shocks**. Throughout the analysis, I use the **general (median) decomposition** of Jarociński and Karadi (2020), which allows both types of shocks to occur jointly within a given announcement.

This specification is essential for analyzing equity duration, as it permits a clean separation between **discount-rate effects** and **cash-flow or information effects**.

---

#### Two-shock interaction regressions

The baseline empirical specification is:

$$
AR_{i,t}
=
\beta_1 \big( ShockMP_t \times D_{i,t-1} \big)
+
\beta_2 \big( ShockInfo_t \times D_{i,t-1} \big)
+
\Gamma' X_{i,t-1}
+
\tau_t
+
\varepsilon_{i,t},
$$

where $AR_{i,t}$ denotes the abnormal return of firm $i$ on ECB announcement date $t$,  
$D_{i,t-1}$ is a standardized, ex-ante equity duration measure,  
and $X_{i,t-1}$ includes firm-level controls such as size and market beta.  

Event fixed effects $\tau_t$ absorb the average market reaction to each announcement, so identification comes from **cross-sectional heterogeneity in duration within each event**.

---

#### Core hypotheses and expected signs

**Monetary policy shocks (`ShockMP`)**

A positive monetary policy shock corresponds to an unexpected tightening of policy and an increase in discount rates. Since high-duration equities derive a larger share of their value from distant cash flows, they should be more sensitive to such shocks.

- **Hypothesis 1 (Discount-rate channel):**
  $$
  \beta_1 < 0
  $$
  High-duration firms experience more negative abnormal returns following tightening surprises.

**Information shocks (`ShockInfo`)**

Information shocks capture news about the economic outlook conveyed by the ECB that affects expected cash flows rather than discount rates. The duration implication is therefore ambiguous ex ante.

- **Hypothesis 2 (Information / cash-flow channel):**  
  The sign of $\beta_2$ is an empirical question.
  - If information shocks primarily revise long-run growth expectations, long-duration firms may react more strongly ($\beta_2 > 0$).
  - If information shocks mainly affect near-term cash flows or cyclical firms, duration may play a limited role ($\beta_2 \approx 0$).

---

#### Equity duration measures

The primary duration measure is a **net-payout-based equity duration**, which infers the timing of shareholder cash flows from market valuations and expected net payouts. This measure is forward-looking, internally consistent with observed prices, and well suited for analyzing discount-rate sensitivity.

To assess robustness and disentangle underlying mechanisms, I compare results to two benchmark duration measures:

- **`Duration_CAPM`**: a DCF-based duration using firm-specific CAPM-implied discount rates, capturing both cash-flow timing and heterogeneity in required returns.
- **`Duration_r125`**: a DCF-based duration using a common discount rate of 12.5%, capturing primarily cash-flow timing.

All duration measures are standardized prior to estimation, and results are reported side-by-side across measures.

---

#### Inference and interpretation

Standard errors are **clustered by event date**, reflecting the fact that monetary policy and information shocks are common across firms within an ECB announcement.

The coefficients of interest are:
- `ShockMP × Duration`: heterogeneity in **discount-rate sensitivity** across firms.
- `ShockInfo × Duration`: heterogeneity in responses to **information and cash-flow news**.

Even if average market reactions are modest, statistically significant differences across duration measures indicate meaningful cross-sectional heterogeneity in monetary policy transmission to equity prices.

In [36]:
# ============================================================
# BASELINE REGRESSION
# Two-shock model with event fixed effects
# - Controls: log(ME), BETA_5Y
# - Excludes Financials (EconomicSector == "Financials")
# ============================================================

# ------------------------------------------------------------
# 0) Choose baseline standardized duration
# ------------------------------------------------------------
BASELINE_DUR_STD = "Duration_NP_std"
assert BASELINE_DUR_STD in df_evt.columns, f"{BASELINE_DUR_STD} not found in df_evt."

# ------------------------------------------------------------
# 1) Drop Financials (time-invariant sector classification)
# ------------------------------------------------------------
assert "EconomicSector" in df_evt.columns, "EconomicSector not found in df_evt."

df_evt = df_evt[df_evt["EconomicSector"] != "Financials"].copy()

print("Dropped Financials. Remaining observations:", df_evt.shape)

# ------------------------------------------------------------
# 2) Controls: log(ME) + standardize controls
# ------------------------------------------------------------
assert "ME" in df_evt.columns, "ME not found in df_evt."
assert "BETA_5Y" in df_evt.columns, "BETA_5Y not found in df_evt."

df_evt["log_ME"] = np.log(df_evt["ME"])

for c in ["BETA_5Y", "log_ME"]:
    mu = df_evt[c].mean(skipna=True)
    sd = df_evt[c].std(skipna=True, ddof=0)
    df_evt[f"{c}_std"] = (df_evt[c] - mu) / sd if (pd.notna(sd) and sd != 0) else pd.NA

# ------------------------------------------------------------
# 3) Build regression sample
# ------------------------------------------------------------
reg_cols = [
    "AR", "ShockMP", "ShockInfo", BASELINE_DUR_STD, "date",
    "BETA_5Y_std", "log_ME_std"
]
df_reg = df_evt[reg_cols].dropna().copy()

print("Baseline regression sample:", df_reg.shape)
print("Unique event dates:", df_reg["date"].nunique())

# ------------------------------------------------------------
# 4) Baseline formula (event FE absorb shock main effects)
# ------------------------------------------------------------
formula = (
    f"AR ~ ShockMP:{BASELINE_DUR_STD} + ShockInfo:{BASELINE_DUR_STD} "
    f"+ ShockMP:BETA_5Y_std + ShockInfo:BETA_5Y_std "
    f"+ ShockMP:log_ME_std + ShockInfo:log_ME_std "
    f"+ C(date)"
)

print("Formula:", formula)

# ------------------------------------------------------------
# 5) Estimate with event-date clustered SEs
# ------------------------------------------------------------
m_baseline = smf.ols(formula, data=df_reg).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_reg["date"]}
)

# ------------------------------------------------------------
# 6) Display economically relevant coefficients
# ------------------------------------------------------------
res = pd.DataFrame({
    "coef": m_baseline.params,
    "std_err": m_baseline.bse,
    "t": m_baseline.tvalues,
    "pvalue": m_baseline.pvalues
})

display(
    res.loc[
        res.index.str.contains(
            r"Shock(MP|Info):|BETA_5Y_std|log_ME_std", regex=True
        )
    ]
)

Dropped Financials. Remaining observations: (216498, 25)
Baseline regression sample: (82434, 7)
Unique event dates: 265
Formula: AR ~ ShockMP:Duration_NP_std + ShockInfo:Duration_NP_std + ShockMP:BETA_5Y_std + ShockInfo:BETA_5Y_std + ShockMP:log_ME_std + ShockInfo:log_ME_std + C(date)


/var/folders/r2/vtz74sz14fx185wt9m3c781w0000gn/T/ipykernel_44630/2173485978.py:80: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  res.index.str.contains(


,coef,std_err,t,pvalue
ShockMP:Duration_NP_std,0.098970,0.504658,0.196114,0.844521
ShockInfo:Duration_NP_std,0.918456,0.557328,1.647962,0.099360
ShockMP:BETA_5Y_std,-3.663562,1.389419,-2.636759,0.008370
ShockInfo:BETA_5Y_std,5.745914,1.278049,4.495849,0.000007
ShockMP:log_ME_std,-2.866592,0.676270,-4.238829,0.000022
ShockInfo:log_ME_std,1.625990,0.872234,1.864167,0.062298


## 8. Robustness: [0, +1] event window (two-day cumulative abnormal returns)

As a robustness check, I extend the event window to include the trading day following the ECB announcement and compute two-day cumulative abnormal returns:

$$
AR^{0,1}_{i,t} = AR_{i,t} + AR_{i,t+1}.
$$

This specification captures potential **delayed market reactions** and **post-announcement information digestion**, which may be particularly relevant for information shocks conveyed during the press conference.

**Implementation details:**
- The day $t+1$ return corresponds to the **next trading day for the same firm**, constructed by shifting abnormal returns within each firm (`groupby("RIC")`).
- If the next-day return is unavailable (e.g., due to end-of-sample truncation or delistings), the cumulative return is set to missing.
- Observations with missing $AR^{0,1}_{i,t}$ are dropped explicitly prior to estimation to ensure a consistent event-level clustering of standard errors.

The two-day cumulative return is then analyzed using the same **two-shock regression framework with event fixed effects** as in the baseline specification. This ensures that identification continues to rely on **cross-sectional heterogeneity within each ECB announcement**, while allowing for a more flexible adjustment horizon of equity prices.

In [37]:
# ============================================================
# 8. Robustness: [0,+1] abnormal return window
# - builds AR_01
# - merges shocks (date)
# - merges predetermined firm-year vars (RIC x YEAR): durations + ME + BETA_5Y
# - merges sectors by RIC only
# - excludes Financials
# - standardizes durations + controls within df_evt_01
# - runs two-shock regression with event FE + clustered SEs
# ============================================================

# --- 0) Build AR[0,+1] ---
df_ret_sorted = df_ret.copy()
df_ret_sorted["date"] = pd.to_datetime(df_ret_sorted["date"], errors="coerce")
df_ret_sorted = df_ret_sorted.sort_values(["RIC", "date"])

df_ret_sorted["AR0"] = pd.to_numeric(df_ret_sorted["AR"], errors="coerce")
df_ret_sorted["AR1"] = df_ret_sorted.groupby("RIC")["AR0"].shift(-1)
df_ret_sorted["AR_01"] = df_ret_sorted["AR0"] + df_ret_sorted["AR1"]

# --- 1) Keep only ECB event dates (day 0) ---
df_shk["date"] = pd.to_datetime(df_shk["date"], errors="coerce")
event_dates = df_shk["date"].dropna().unique()

df_evt_01 = df_ret_sorted[df_ret_sorted["date"].isin(event_dates)].copy()
df_evt_01["RIC"] = df_evt_01["RIC"].astype(str).str.strip()
print("Event-panel [0,+1] shape (before merges):", df_evt_01.shape)

# --- 2) Merge shocks (m:1) ---
df_evt_01 = df_evt_01.merge(
    df_shk[["date", "ShockMP", "ShockInfo"]],
    on="date",
    how="left",
    validate="m:1"
)
miss_shk = df_evt_01[["ShockMP", "ShockInfo"]].isna().any(axis=1).mean()
print("Share of rows with missing shocks:", miss_shk)
assert miss_shk == 0, "Some event rows could not be matched to shocks."

# --- 3) Predetermined YEAR and merge firm-year vars (m:1) ---
df_evt_01["YEAR"] = (df_evt_01["date"].dt.year - 1).astype("Int64")

df_dur_panel_use = df_dur_panel.copy()
df_dur_panel_use["RIC"] = df_dur_panel_use["RIC"].astype(str).str.strip()

# harmonize year column in df_dur_panel_use
if "YEAR" not in df_dur_panel_use.columns and "year" in df_dur_panel_use.columns:
    df_dur_panel_use = df_dur_panel_use.rename(columns={"year": "YEAR"})
df_dur_panel_use["YEAR"] = pd.to_numeric(df_dur_panel_use["YEAR"], errors="coerce").astype("Int64")

duration_cols = [c for c in df_dur_panel_use.columns if c.startswith("Duration_")]
if not duration_cols:
    duration_cols = [c for c in df_dur_panel_use.columns if c.startswith("D") and c not in ["DATE", "DAY", "RIC", "YEAR"]]
if not duration_cols:
    raise ValueError("No duration columns found in df_dur_panel (expected 'Duration_*' or 'D*').")

num_cols = [c for c in (duration_cols + ["ME", "BETA_5Y"]) if c in df_dur_panel_use.columns]
for c in num_cols:
    df_dur_panel_use[c] = pd.to_numeric(df_dur_panel_use[c], errors="coerce")

df_evt_01 = df_evt_01.merge(
    df_dur_panel_use[["RIC", "YEAR"] + num_cols].drop_duplicates(["RIC", "YEAR"]),
    on=["RIC", "YEAR"],
    how="left",
    validate="m:1"
)

# sectors by RIC only
sector_cols = [c for c in ["EconomicSector", "BusinessSector"] if c in df_dur_panel_use.columns]
sector_map = df_dur_panel_use[["RIC"] + sector_cols].drop_duplicates("RIC")
df_evt_01 = df_evt_01.merge(sector_map, on="RIC", how="left", validate="m:1")

# exclude Financials
df_evt_01 = df_evt_01[df_evt_01["EconomicSector"] != "Financials"].copy()
print("Dropped Financials. Remaining observations:", df_evt_01.shape)

# --- 4) Controls: log(ME) + standardize controls ---
df_evt_01["log_ME"] = np.log(df_evt_01["ME"])

for c in ["BETA_5Y", "log_ME"]:
    mu = df_evt_01[c].mean(skipna=True)
    sd = df_evt_01[c].std(skipna=True, ddof=0)
    df_evt_01[f"{c}_std"] = (df_evt_01[c] - mu) / sd if (pd.notna(sd) and sd != 0) else pd.NA

# --- 5) Standardize durations within this panel ---
std_cols = []
for c in duration_cols:
    mu = df_evt_01[c].mean(skipna=True)
    sd = df_evt_01[c].std(skipna=True, ddof=0)
    new_c = f"{c}_std"
    std_cols.append(new_c)
    df_evt_01[new_c] = (df_evt_01[c] - mu) / sd if (pd.notna(sd) and sd != 0) else pd.NA

print("Standardized duration columns added (first 10):", std_cols[:10])

# --- 6) Regression-ready sample ---
BASELINE_DUR_STD = "Duration_NP_std"
if BASELINE_DUR_STD not in df_evt_01.columns:
    raise ValueError(f"{BASELINE_DUR_STD} not found. Available std cols include: {std_cols[:15]} ...")

reg_cols_01 = [
    "AR_01", "ShockMP", "ShockInfo", BASELINE_DUR_STD, "date",
    "BETA_5Y_std", "log_ME_std"
]
df_reg_01 = df_evt_01[reg_cols_01].dropna().copy()

print("Regression sample [0,+1]:", df_reg_01.shape)
print("Unique event dates:", df_reg_01["date"].nunique())

# --- 7) Two-shock regression with event FE (controls interacted with shocks) ---
formula_01 = (
    f"AR_01 ~ ShockMP:{BASELINE_DUR_STD} + ShockInfo:{BASELINE_DUR_STD} "
    f"+ ShockMP:BETA_5Y_std + ShockInfo:BETA_5Y_std "
    f"+ ShockMP:log_ME_std + ShockInfo:log_ME_std "
    f"+ C(date)"
)
print("Formula:", formula_01)

m2_01 = smf.ols(formula_01, data=df_reg_01).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_reg_01["date"]}
)

res_01 = pd.DataFrame({
    "coef": m2_01.params,
    "std_err": m2_01.bse,
    "t": m2_01.tvalues,
    "pvalue": m2_01.pvalues,
})

display(
    res_01.loc[
        res_01.index.str.contains(r"Shock(MP|Info):|BETA_5Y_std|log_ME_std", regex=True)
    ]
)

Event-panel [0,+1] shape (before merges): (226233, 8)
Share of rows with missing shocks: 0.0
Dropped Financials. Remaining observations: (216498, 20)
Standardized duration columns added (first 10): ['Duration_NP_std', 'Duration_NP_w_std', 'Duration_undiscounted_std', 'Duration_r125_std', 'Duration_CAPM_std', 'Duration_emp_2yOIS_std']
Regression sample [0,+1]: (156772, 7)
Unique event dates: 265
Formula: AR_01 ~ ShockMP:Duration_NP_std + ShockInfo:Duration_NP_std + ShockMP:BETA_5Y_std + ShockInfo:BETA_5Y_std + ShockMP:log_ME_std + ShockInfo:log_ME_std + C(date)


/var/folders/r2/vtz74sz14fx185wt9m3c781w0000gn/T/ipykernel_44630/3782279276.py:133: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  res_01.index.str.contains(r"Shock(MP|Info):|BETA_5Y_std|log_ME_std", regex=True)


,coef,std_err,t,pvalue
ShockMP:Duration_NP_std,-0.270825,0.509379,-0.531677,0.594950
ShockInfo:Duration_NP_std,0.935377,0.688652,1.358272,0.174377
ShockMP:BETA_5Y_std,-4.338561,1.393199,-3.114099,0.001845
ShockInfo:BETA_5Y_std,6.572229,1.613078,4.074340,0.000046
ShockMP:log_ME_std,-1.218998,0.761239,-1.601334,0.109303
ShockInfo:log_ME_std,1.126423,0.748620,1.504666,0.132410


## 9. Portfolio-split regressions (non-linearity)

To allow for potential **non-linear duration effects**, I complement the continuous interaction regressions with **portfolio-split specifications** that compare firms with very different equity durations.

### Portfolio construction

For each calendar year, firms are sorted into terciles based on their **predetermined equity duration**.  
I then retain only firms in the **lowest** and **highest** terciles and define a binary indicator:

- `HighDur_i = 1` if firm $i$ belongs to the **high-duration tercile**,
- `HighDur_i = 0` if firm $i$ belongs to the **low-duration tercile**.

The middle tercile is excluded to sharpen the contrast between short- and long-duration firms.

---

### Two-shock portfolio-split regression

Using the same event-study framework as in the baseline analysis, I estimate the following regression:

$$
AR_{i,t}
=
\beta_1 \big( ShockMP_t \times HighDur_i \big)
+
\beta_2 \big( ShockInfo_t \times HighDur_i \big)
+
\tau_t
+
\varepsilon_{i,t},
$$

where $AR_{i,t}$ denotes the abnormal return of firm $i$ on ECB announcement date $t$, and $\tau_t$ are event fixed effects.

Because shocks are constant within each event, the inclusion of $\tau_t$ absorbs the average market reaction to the announcement. Identification therefore comes from **within-event differences** between high- and low-duration firms.

---

### Interpretation

The interaction terms have a direct and intuitive interpretation:

- `ShockMP × HighDur` measures how much **more (or less)** high-duration firms react to **monetary policy shocks** relative to low-duration firms.
- `ShockInfo × HighDur` measures differential responses to **information shocks** conveyed by the ECB.

This specification provides a transparent test for **non-linearities** in duration effects. Even if continuous interaction coefficients are small or imprecisely estimated, economically meaningful differences between high- and low-duration firms may still emerge in the tails of the duration distribution.

The portfolio-split results therefore serve as a complementary robustness check to the continuous interaction regressions.

In [38]:
# ============================================================
# Portfolio split (robust): year-by-year QUINTILES (Q1 vs Q5)
# - excludes Financials
# - controls: log(ME), BETA_5Y (interacted with shocks)
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

df_evt = df_evt.copy()
df_evt["date"] = pd.to_datetime(df_evt["date"], errors="coerce")

# ------------------------------------------------------------
# 0) Exclude Financials
# ------------------------------------------------------------
assert "EconomicSector" in df_evt.columns, "EconomicSector missing in df_evt."
df_evt = df_evt[df_evt["EconomicSector"] != "Financials"].copy()

print("After excluding Financials:", df_evt.shape)

# ------------------------------------------------------------
# 1) Predetermined YEAR (t-1, consistent with duration merge)
# ------------------------------------------------------------
if "YEAR" not in df_evt.columns:
    df_evt["YEAR"] = (df_evt["date"].dt.year - 1).astype("Int64")

# ------------------------------------------------------------
# 2) Choose duration for sorting (raw, predetermined)
# ------------------------------------------------------------
BASE_DUR = "Duration_NP"   # or Duration_CAPM, Duration_r125
assert BASE_DUR in df_evt.columns, f"{BASE_DUR} not found in df_evt."

def safe_qcut_bins(x: pd.Series, q: int = 5, min_n: int = 100) -> pd.Series:
    x = pd.to_numeric(x, errors="coerce")
    ok = x.notna()
    if ok.sum() < min_n:
        return pd.Series(pd.NA, index=x.index, dtype="object")

    r = x[ok].rank(method="average")
    try:
        labels = [f"Q{k}" for k in range(1, q + 1)]
        b = pd.qcut(r, q=q, labels=labels)
        out = pd.Series(pd.NA, index=x.index, dtype="object")
        out.loc[b.index] = b.astype("object")
        return out
    except Exception:
        return pd.Series(pd.NA, index=x.index, dtype="object")

# year-by-year quintiles
df_evt["Dur_bin"] = (
    df_evt.groupby("YEAR")[BASE_DUR]
          .transform(lambda s: safe_qcut_bins(s, q=5, min_n=100))
)

print("Dur_bin counts:")
display(df_evt["Dur_bin"].value_counts(dropna=False))

# ------------------------------------------------------------
# 3) Keep Q1 vs Q5
# ------------------------------------------------------------
df_evt_q15 = df_evt[df_evt["Dur_bin"].isin(["Q1", "Q5"])].copy()
df_evt_q15["HighDur"] = (df_evt_q15["Dur_bin"] == "Q5").astype(int)

print("Q1/Q5 sample shape:", df_evt_q15.shape)
display(df_evt_q15["Dur_bin"].value_counts())

# ------------------------------------------------------------
# 4) Controls: log(ME) + standardize controls
# ------------------------------------------------------------
assert "ME" in df_evt_q15.columns, "ME missing in df_evt."
assert "BETA_5Y" in df_evt_q15.columns, "BETA_5Y missing in df_evt."

df_evt_q15["log_ME"] = np.log(df_evt_q15["ME"])

for c in ["BETA_5Y", "log_ME"]:
    mu = df_evt_q15[c].mean(skipna=True)
    sd = df_evt_q15[c].std(skipna=True, ddof=0)
    df_evt_q15[f"{c}_std"] = (df_evt_q15[c] - mu) / sd if (pd.notna(sd) and sd != 0) else pd.NA

# ------------------------------------------------------------
# 5) Regression sample
# ------------------------------------------------------------
reg_cols = [
    "AR", "ShockMP", "ShockInfo", "HighDur", "date",
    "BETA_5Y_std", "log_ME_std"
]

df_reg_ps = df_evt_q15[reg_cols].dropna().copy()

print("Regression sample (Q1 vs Q5):", df_reg_ps.shape)
print("Unique event dates:", df_reg_ps["date"].nunique())

# ------------------------------------------------------------
# 6) Two-shock portfolio-split regression with event FE
# ------------------------------------------------------------
formula = (
    "AR ~ ShockMP:HighDur + ShockInfo:HighDur "
    "+ ShockMP:BETA_5Y_std + ShockInfo:BETA_5Y_std "
    "+ ShockMP:log_ME_std + ShockInfo:log_ME_std "
    "+ C(date)"
)

print("Formula:", formula)

m_ps = smf.ols(formula, data=df_reg_ps).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_reg_ps["date"]}
)

# ------------------------------------------------------------
# 7) Display relevant coefficients
# ------------------------------------------------------------
res = pd.DataFrame({
    "coef": m_ps.params,
    "std_err": m_ps.bse,
    "t": m_ps.tvalues,
    "pvalue": m_ps.pvalues
})

display(
    res.loc[
        res.index.str.contains(
            r"Shock(MP|Info):HighDur|BETA_5Y_std|log_ME_std",
            regex=True
        )
    ]
)

After excluding Financials: (216498, 28)
Dur_bin counts:


Dur_bin
<NA>    127223
Q1       17966
Q2       17886
Q4       17864
Q3       17843
Q5       17716
Name: count, dtype: int64

Q1/Q5 sample shape: (35682, 30)


Dur_bin
Q1    17966
Q5    17716
Name: count, dtype: int64

Regression sample (Q1 vs Q5): (32744, 7)
Unique event dates: 265
Formula: AR ~ ShockMP:HighDur + ShockInfo:HighDur + ShockMP:BETA_5Y_std + ShockInfo:BETA_5Y_std + ShockMP:log_ME_std + ShockInfo:log_ME_std + C(date)


/var/folders/r2/vtz74sz14fx185wt9m3c781w0000gn/T/ipykernel_44630/114227816.py:123: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  res.index.str.contains(


,coef,std_err,t,pvalue
ShockMP:HighDur,0.430603,1.264836,0.340441,7.335241e-01
ShockInfo:HighDur,1.486182,1.527769,0.972779,3.306633e-01
ShockMP:BETA_5Y_std,-3.486664,1.350256,-2.582225,9.816551e-03
ShockInfo:BETA_5Y_std,5.814235,1.184889,4.906987,9.248612e-07
ShockMP:log_ME_std,-2.272306,0.681523,-3.334159,8.555764e-04
ShockInfo:log_ME_std,1.146995,0.879500,1.304145,1.921843e-01
